# Data Cleaning and EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load the Data

In [ ]:
url = "http://ddc-datascience.s3.amazonaws.com/winequality-red.csv"
wine_data = pd.read_csv(url)

In [ ]:
wine_data.shape

In [ ]:
wine_data.size

In [ ]:
f"{wine_data.size:,}"

## Understand the Data

In [ ]:
# Look at the first five rows
wine_data.head()

In [ ]:
# Get info about each column
wine_data.info()

In [ ]:
# Count the number of nulls then multiply by some large number to make them stand out
wine_data.isnull().sum()*1000

In [ ]:
# Count the number of nulls then sort to group counts
wine_data.isnull().sum().sort_values()

Notice that there is one null vlaue for pH and residual sugar.

In [ ]:
# Get summary statistics about each column
wine_data.describe()

In [ ]:
type(wine_data.describe())

In [ ]:
wine_data.describe().transpose()

In [ ]:
# See the unique values in the quality column
wine_data['quality'].unique()

In [ ]:
# Count the number of unique values in the quality column
wine_data['quality'].nunique()

In [ ]:
len(wine_data['quality'].unique())

In [ ]:
# See how many observations there are for each unique value in the quality column
wine_data['quality'].value_counts()

In [ ]:
wine_data['quality'].value_counts(normalize=True)*100

## Clean the Data

We saw above that there are two columns that have nulls/NAs. Let's drop the rows that have NAs.

In [ ]:
# Before doing any cleaning, let's make a copy of our data frame
wine_clean = wine_data.copy()
wine_clean


In [ ]:
# Drop the rows with NAs
wine_clean.dropna(inplace = True)
wine_clean


In [ ]:
# Alternate, doing both steps in one
# wine_clean = wine_data.dropna()

In [ ]:
# Verify they NAs were dropped
wine_clean.isnull().sum()*1000

Let's say we don't want to include the density or sulphate columns in our data, so we will drop them.

In [ ]:
# View column names
wine_clean.columns


In [ ]:
# Drop the density and sulphate column
wine_clean.drop(columns = ['density', 'sulphates'], inplace = True)

In [ ]:
# Verify it was dropped
wine_clean.columns


In [ ]:
# Sort list using Python functions
list(wine_clean.columns)

In [ ]:
# Sort list using Python functions
sorted(list(wine_clean.columns))

In [ ]:
# Sort list using Pandas methods
wine_clean.columns.sort_values().to_list()

In [ ]:
wine_clean.isnull().sum()*1000

In [ ]:
( wine_clean.isnull().sum()*1000 ).sort_index()

Finally, residual sugar is currently in g but I want it in mg. So I am going to multiply each value in the residual sugar column by 1000.

In [ ]:
wine_clean['residual sugar'].head()

In [ ]:
# Multiply each value by 1000
wine_clean['residual sugar (mg)'] = wine_clean['residual sugar']*1000


In [ ]:
# Confirm it worked
wine_clean[['residual sugar','residual sugar (mg)']].head()

In [ ]:
wine_clean['residual sugar'] = wine_clean['residual sugar (mg)']
wine_clean.head()

In [ ]:
wine_clean.drop( columns=["residual sugar (mg)"], inplace=True)

In [ ]:
wine_clean.head()

## Generate Summary Statistics

In [ ]:
# Get summary statistics for each column
wine_clean.describe()

In [ ]:
wine_clean.describe().transpose()

In [ ]:
wine_clean.describe().transpose().loc["alcohol"]["mean"]

In [ ]:
# Calculate the mean of a single column
wine_clean['alcohol'].mean()

## Plot the data (univariate)

In [ ]:
wine_clean['alcohol'].hist()
plt.xlabel('% Alcohol')
plt.ylabel('Count') ;

In [ ]:
wine_clean['alcohol'].hist(bins = 30)
plt.xlabel('% Alcohol')
plt.ylabel('Count') ;

In [ ]:
sns.histplot(x = wine_clean['alcohol'], kde = True) ;

In [ ]:
wine_clean.boxplot(column = 'citric acid') ;

What's going on here? It looks like there might be a large outlier. Could we have identified without graphing? Let's take a look.



In [ ]:
wine_clean_stats = wine_clean.describe().transpose()
wine_clean_stats


In [ ]:
# Range stats
wine_clean_stats["IQR"]     = ( wine_clean_stats["75%"]  - wine_clean_stats["25%"] )
wine_clean_stats["min_eff"] = ( wine_clean_stats["25%"]  - wine_clean_stats["IQR"] * 1.5 )
wine_clean_stats["max_eff"] = ( wine_clean_stats["75%"]  + wine_clean_stats["IQR"] * 1.5 )
wine_clean_stats["std_-2"]  = ( wine_clean_stats["mean"] - wine_clean_stats["std"] * 2 )
wine_clean_stats["std_+2"]  = ( wine_clean_stats["mean"] + wine_clean_stats["std"] * 2 )
wine_clean_stats["std_-3"]  = ( wine_clean_stats["mean"] - wine_clean_stats["std"] * 3 )
wine_clean_stats["std_+3"]  = ( wine_clean_stats["mean"] + wine_clean_stats["std"] * 3 )
wine_clean_stats


In [ ]:
# Above effective max
wine_clean_stats["max"] > wine_clean_stats["max_eff"]


In [ ]:
wine_clean_stats["max"] > wine_clean_stats["std_+3"]


In [ ]:
wine_clean_stats["max"] > ( wine_clean_stats["mean"] + wine_clean_stats["std"] * 12 )


In [ ]:
# Find the outlier
ca_max = wine_clean['citric acid'].max()
ca_max

In [ ]:
filter = ( wine_clean['citric acid'] == ca_max )
filter


In [ ]:
wine_clean[filter]

In [ ]:
# Look at other rows to get a feel for "normal"
wine_clean.head()

The citric acid in this row is 100. All the other citric acid values are much smaller. We talked to the person who put together this dataset and they confirmed this is a mistake. Let's replace this value with the mean citric acid value.

## Impute outlier

In [ ]:
wine_clean_drop = wine_clean.copy()
wine_clean_drop.head()


In [ ]:
wine_clean_drop[filter][['citric acid']]


In [ ]:
# Calculate mean, std with outlier
( wine_clean_drop['citric acid'].mean(), wine_clean_drop['citric acid'].std() )


In [ ]:
# Change max value to NaN
wine_clean_drop.loc[filter, 'citric acid'] = np.nan
wine_clean_drop[filter][['citric acid']]


In [ ]:
# Calculate the mean, std of citric acid
ca_mean, _ = wine_clean_drop['citric acid'].mean(), wine_clean_drop['citric acid'].std()
ca_mean, _


In [ ]:
# Replace null with mean
wine_clean_drop.loc[filter, 'citric acid'] = ca_mean
wine_clean_drop[filter][['citric acid']]


In [ ]:
# Calculate mean, std with imputed value
( wine_clean_drop['citric acid'].mean(), wine_clean_drop['citric acid'].std(), )

In [ ]:
sns.histplot( x=wine_clean_drop["citric acid"]) ;

In [ ]:
wine_clean_drop.boxplot(column = 'citric acid') ;

## Plot the data (bivariate)

In [ ]:
# Column correlations
column_correlations = wine_clean_drop.corr()
column_correlations


In [ ]:
type(column_correlations)

In [ ]:
# Correlation Plot
plt.figure(figsize=(10,6))
sns.heatmap(column_correlations, annot=True, vmin = -1, vmax = 1, cmap='RdYlGn') ;

In [ ]:
# Get most highly correlated varliables with alcohol
alc_correlations = column_correlations['alcohol']
alc_correlations

In [ ]:
correlated_vars = (
  alc_correlations
  .abs()
  .sort_values(ascending=False)
  .drop("alcohol")
)
correlated_vars

In [ ]:
# Scatter plot
wine_clean_drop.plot(x='fixed acidity', y='pH', kind='scatter',  title='pH vs Fixed Acidity') ;

In [ ]:
# Scatter plot with different colors based on quality
sns.scatterplot(
    x    = wine_clean_drop['alcohol'],
    y    = wine_clean_drop['pH'],
    hue  = wine_clean_drop['quality'],
    size = wine_clean_drop['quality'],
) ;

In [ ]:
# Pairplot
sns.pairplot(wine_clean_drop) ;

In [ ]:
wine_clean_drop.hist() ;